In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY") 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [2]:
# Data ingestion --> from the website we need to scrape the data 
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://www.geeksforgeeks.org/dsa/array-data-structure-guide/")
docs = loader.load()
   

C:\Users\divya\AppData\Local\Temp\ipykernel_21688\910747705.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [11]:
# Load data --> Docs --> Divide data into chunks --> vectors --> vector embeddings--> vector store db
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap = 200)
documents = text_splitter.split_documents(docs)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vectorstoredb = FAISS.from_documents(documents,embeddings)
vectorstoredb

In [ ]:
query = "Provide me important DSA Concepts"
result = vectorstoredb.similarity_search(query)[0].page_content
result




'Basics Introduction   Applications  In Different Language Arrays in C Vector in C++ STL  Arrays in Java  ArrayList in Java List in Python Arrays in C#  Arrays in JavaScript   Basic Problems  Print AlternatesLeaders in an array  Remove Duplicates from Sorted Generate all Subarrays Reverse an Array Rotate an Array Zeroes to End  Min Increments to Make Equal  Min Cost to Make Size 1 Easy Problems Duplicate within K DistanceMake Even Positioned GreaterSum of all SubarraysStock Buy and Sell – Multiple TransactionsSingle Among DoublesMissing NumberMissing and RepeatingOnly Repeating from 1 to n-1Sorted Subsequence of Size 3 Max Subarray SumEquilibrium index  Split array into three equals  Prerequisite for the Remaining ProblemsBinary SearchSelection Sort, Insertion Sort, Binary Search, QuickSort, MergeSort, CycleSort, and HeapSortSort in C++  /  Sort in Java / Sort in Python / Sort in JavaScriptTwo Pointers Technique Prefix Sum TechniqueBasics of Hashing Window Sliding Technique Medium'

In [24]:
# Retrieval chain,document chain 
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate 
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=1.0,  
    max_tokens=None,
    timeout=None,
    max_retries=2
)

prompt = ChatPromptTemplate.from_template(
    """ Answer the following questions based on the provided context: 
    <context>
    {context}
    </context>
    """
)
document_chain = create_stuff_documents_chain(llm,prompt)
print(document_chain)



bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' Answer the following questions based on the provided context: \n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.3'}}, output_version=None, profile={'name': 'Gemini 3.6 Flash', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'im

In [25]:
document_chain.invoke({
    "input":"ndian development is not so good in terms of infrasture here",
    "context":[Document(page_content = "In recent years world has experienced global crisis and indian development is not so good in terms of infrasture here")]
})

C:\Users\divya\AppData\Roaming\Python\Python314\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


"It looks like you didn't include the specific question(s) you would like answered! \n\nBased on the context provided, here are the main points stated:\n1. The world has experienced a global crisis in recent years.\n2. Indian development is not very good in terms of infrastructure.\n\nPlease reply with your question(s), and I will gladly answer them based on this text."

In [ ]:
# However we want the document to come up before retriever is set up. 
# input --> retriever --> vectorstoredb 
# we convert vector store db to retriever 
retriever =vectorstoredb.as_retriever()

retrieval_chain = create_retrieval_chain(retriever,document_chain)


In [29]:
# Get response from the llm 
retrieval_chain.invoke({"input":"Rotate an array"})

C:\Users\divya\AppData\Roaming\Python\Python314\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'input': 'Rotate an array',
 'context': [Document(id='849a8eb0-1b3a-4bdc-9edd-0009e190c6a0', metadata={'source': 'https://www.geeksforgeeks.org/dsa/array-data-structure-guide/', 'title': 'Array Data Structure - GeeksforGeeks', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en'}, page_content='Basics Introduction   Applications  In Different Language Arrays in C Vector in C++ STL  Arrays in Java  ArrayList in Java List in Python Arrays in C#  Arrays in JavaScript   Basic Problems  Print AlternatesLeaders in an array  Remove Duplicates from Sorted Generate all Subarrays Reverse an Array Rotate an Array Zeroes to End  Min Increments to Make Equal  Min Cost to Make Size 1 Easy Problems Duplicate within K DistanceMake Even Positioned GreaterSum of all Subar